In [ ]:
!nvidia-smi


Mon Jun 29 04:43:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile main.cu
#include <iostream>
#include <vector>
#include <cmath>
#include <random>
#include <chrono>
#include <thread>
#include <curand_kernel.h>

struct OptionParams {
    float S0, K, r, sigma, T;
};

__global__ void monte_carlo_kernel(
    OptionParams p,
    float* partial_sums,
    int paths,
    unsigned long seed
) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int total_threads = blockDim.x * gridDim.x;

    curandState state;
    curand_init(seed, tid, 0, &state);

    float local_sum = 0.0f;

    for (int i = tid; i < paths; i += total_threads) {
        float Z = curand_normal(&state);

        float ST = p.S0 * expf(
            (p.r - 0.5f * p.sigma * p.sigma) * p.T +
            p.sigma * sqrtf(p.T) * Z
        );

        local_sum += fmaxf(ST - p.K, 0.0f);
    }

    partial_sums[tid] = local_sum;
}

double cpu_monte_carlo(const OptionParams& p, int paths) {
    std::mt19937 gen(42);
    std::normal_distribution<float> dist(0.0f, 1.0f);

    double sum = 0.0;

    for (int i = 0; i < paths; i++) {
        float Z = dist(gen);

        float ST = p.S0 * expf(
            (p.r - 0.5f * p.sigma * p.sigma) * p.T +
            p.sigma * sqrtf(p.T) * Z
        );

        sum += fmaxf(ST - p.K, 0.0f);
    }

    return exp(-p.r * p.T) * sum / paths;
}

double gpu_monte_carlo(const OptionParams& p, int paths, int blocks, int threads) {
    int total_threads = blocks * threads;

    float* d_partial_sums;
    cudaMalloc(&d_partial_sums, total_threads * sizeof(float));

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    monte_carlo_kernel<<<blocks, threads>>>(p, d_partial_sums, paths, 1234);

    cudaDeviceSynchronize();

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    std::vector<float> partial_sums(total_threads);

    cudaMemcpy(
        partial_sums.data(),
        d_partial_sums,
        total_threads * sizeof(float),
        cudaMemcpyDeviceToHost
    );

    double sum = 0.0;
    for (float x : partial_sums) {
        sum += x;
    }

    float kernel_ms = 0.0f;
    cudaEventElapsedTime(&kernel_ms, start, stop);

    cudaFree(d_partial_sums);

    std::cout << "GPU kernel time: " << kernel_ms << " ms\n";

    return exp(-p.r * p.T) * sum / paths;
}

int main() {
    OptionParams params;
    params.S0 = 100.0f;
    params.K = 100.0f;
    params.r = 0.05f;
    params.sigma = 0.2f;
    params.T = 1.0f;

    int paths = 10000000;

    int threads = 256;
    int blocks = 1024;

    auto cpu_start = std::chrono::high_resolution_clock::now();
    double cpu_price = cpu_monte_carlo(params, paths);
    auto cpu_end = std::chrono::high_resolution_clock::now();

    double cpu_ms =
        std::chrono::duration<double, std::milli>(cpu_end - cpu_start).count();

    auto gpu_start = std::chrono::high_resolution_clock::now();
    double gpu_price = gpu_monte_carlo(params, paths, blocks, threads);
    auto gpu_end = std::chrono::high_resolution_clock::now();

    double gpu_ms =
        std::chrono::duration<double, std::milli>(gpu_end - gpu_start).count();

    std::cout << "\nCPU price: " << cpu_price << "\n";
    std::cout << "CPU time: " << cpu_ms << " ms\n";

    std::cout << "\nGPU price: " << gpu_price << "\n";
    std::cout << "GPU total time: " << gpu_ms << " ms\n";

    std::cout << "\nSpeedup: " << cpu_ms / gpu_ms << "x\n";

    return 0;
}

Writing main.cu


In [ ]:
!nvcc -O3 -std=c++17 main.cu -o montecarlo -lcurand

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./montecarlo


GPU kernel time: 268.081 ms

CPU price: 10.4557
CPU time: 479.653 ms

GPU price: 10.4553
GPU total time: 591.723 ms

Speedup: 0.810603x


In [ ]:
!nvprof ./montecarlo

==1516== NVPROF is profiling process 1516, command: ./montecarlo
GPU kernel time: 14.3564 ms

CPU price: 10.4557
CPU time: 373.131 ms

GPU price: 10.4553
GPU total time: 387.777 ms

Speedup: 0.962231x
==1516== Profiling application: ./montecarlo
==1516== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   99.21%  10.195ms         1  10.195ms  10.195ms  10.195ms  monte_carlo_kernel(OptionParams, float*, int, unsigned long)
                    0.79%  81.630us         1  81.630us  81.630us  81.630us  [CUDA memcpy DtoH]
      API calls:   90.52%  176.51ms         1  176.51ms  176.51ms  176.51ms  cudaMalloc
                    5.23%  10.197ms         1  10.197ms  10.197ms  10.197ms  cudaDeviceSynchronize
                    2.12%  4.1402ms         1  4.1402ms  4.1402ms  4.1402ms  cudaLaunchKernel
                    1.29%  2.5057ms       114  21.979us      84ns  1.3297ms  cuDeviceGetAttribute
                    0.61%  1.193